# 02. Feature Engineering

Day 1 전략(2-3, 2-4)에 따른 피처 구성 — 구현: `src/features.py`

| 구분 | 피처 | 근거 |
|---|---|---|
| 셀의 초기 반응 | `log_var_dQ` | ΔQ 통계 중 세 배치 모두 강한 상관, 평균·최솟값과 중복(ρ≈−0.97~−0.99) |
| 원정책 | `C1`, `C2`, `SOC1` | 충전 조건 그대로 |
| 파생정책 | `mean_C`, `max_C` | 속도·피크 요약. `mean_C = 0.8 / [s/C1 + (0.8−s)/C2]` |

In [1]:
# ── 공통 설정 ──────────────────────────────────────────────
import sys
from pathlib import Path

# notebooks/ 에서 실행해도 프로젝트 루트의 src 패키지를 import 할 수 있도록 경로 추가
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 설치된 한글 폰트가 있으면 사용하고, 그림의 영문 표기는 기본 폰트로도 표시한다.
from matplotlib import font_manager
fonts = {f.name for f in font_manager.fontManager.ttflist}
font = next((f for f in ("AppleGothic", "Malgun Gothic", "NanumGothic") if f in fonts), "DejaVu Sans")
sns.set_theme(style="whitegrid", font=font)
plt.rcParams["axes.unicode_minus"] = False

# 배치별 색상 (Day 1 산출물과 동일한 톤)
BATCH_COLOR = {"B1": "#2f6b9a", "B2": "#c77d2a", "B3": "#7f5aa6"}

from src.features import build_features, FEATURE_SETS

# 레이블 보유 셀 피처 테이블
feat = build_features()

# CV에서 비교할 입력 5구성
FEATURE_SETS

{'dQ': ['log_var_dQ'],
 'policy_raw': ['C1', 'C2', 'SOC1'],
 'policy_derived': ['mean_C', 'max_C'],
 'dQ+policy_raw': ['log_var_dQ', 'C1', 'C2', 'SOC1'],
 'dQ+policy_derived': ['log_var_dQ', 'mean_C', 'max_C']}

## 1. ΔQ 사이클·전압축 점검

`src.preprocess`가 summary의 실제 cycle 번호로 배열 인덱스 9·99와 cycle 10·100의 대응을 검사한다. 전압축은 원자료 `Vdlin`(3.5→2.0V)을 사용한다.


In [2]:
from scipy.stats import spearmanr
from src.preprocess import load_cache

b1 = feat[(feat.batch == "B1") & feat.near_eol]
_, _, curves = load_cache()
print("배치별 rho(log var ΔQ, life):",
      {b: round(spearmanr(g.log_var_dQ, g.cycle_life)[0], 2) for b, g in feat.groupby("batch")})
print(f"B1 종료 근접 셀 왜도: var {b1.var_dQ.skew():+.2f} → log {b1.log_var_dQ.skew():+.2f}")
print(f"B1 수명 왜도: {b1.cycle_life.skew():+.2f}")
print({b: (curves[f"voltage|{b}"][0], curves[f"voltage|{b}"][-1]) for b in feat.batch.unique()})


배치별 rho(log var ΔQ, life): {'B1': -0.87, 'B2': -0.71, 'B3': -0.8}
B1 종료 근접 셀 왜도: var +1.11 → log +0.22
B1 수명 왜도: +0.30
{'B1': (3.5, 2.0), 'B2': (3.5, 2.0), 'B3': (3.5, 2.0)}


## 2. 레이블 점검 — B1 46 → 36셀
마지막 QD > 0.885 Ah인 10셀(c00~c04, c08, c10, c12, c13, c22)은 EOL 도달 전에 기록이 끝나 수명 정답으로 쓰지 않음

In [3]:
# 학습에서 제외되는 B1 10셀: 마지막 QD가 EOL(0.88 Ah)보다 한참 높음 → 저장된 cycle_life는 실제 수명이 아님
feat.loc[(feat.batch == "B1") & ~feat.near_eol, ["cell_id", "policy", "cycle_life", "last_QD"]]

,cell_id,policy,cycle_life,last_QD
0,B1_c00,3.6C(80%)-3.6C,1190.0,1.026210
1,B1_c01,3.6C(80%)-3.6C,1179.0,1.038225
2,B1_c02,3.6C(80%)-3.6C,1177.0,1.043279
3,B1_c03,4C(80%)-4C,1226.0,0.970921
4,B1_c04,4C(80%)-4C,1227.0,1.021960
8,B1_c08,5.4C(40%)-3.6C,879.0,0.969025
10,B1_c10,5.4C(50%)-3C,906.0,0.961019
12,B1_c12,5.4C(50%)-3.6C,902.0,0.913119
13,B1_c13,5.4C(50%)-3.6C,897.0,0.923136
22,B1_c22,6C(30%)-3.6C,891.0,0.963279


## 3. 정책 피처와 중복

In [4]:
# 첫 C-rate가 높을수록 전환 SOC가 낮게 설계됨 → 두 변수의 효과를 분리하기 어려움 (Day 1: ρ = -0.88)
print(f"C1 ↔ SOC1 ρ = {spearmanr(b1.C1, b1.SOC1)[0]:.2f}")

# 후보 입력 간 Spearman 상관 (B1에서는 모든 정책이 C1 ≥ C2 라 max_C = C1)
b1[["log_var_dQ", "C1", "C2", "SOC1", "mean_C", "max_C", "cycle_life"]].corr(method="spearman").round(2)

C1 ↔ SOC1 ρ = -0.88


,log_var_dQ,C1,C2,SOC1,mean_C,max_C,cycle_life
log_var_dQ,1.00,0.27,0.12,0.10,0.56,0.27,-0.83
C1,0.27,1.00,-0.10,-0.88,-0.43,1.00,-0.24
C2,0.12,-0.10,1.00,0.05,0.33,-0.10,-0.16
SOC1,0.10,-0.88,0.05,1.00,0.67,-0.88,-0.06
mean_C,0.56,-0.43,0.33,0.67,1.00,-0.43,-0.43
max_C,0.27,1.00,-0.10,-0.88,-0.43,1.00,-0.24
cycle_life,-0.83,-0.24,-0.16,-0.06,-0.43,-0.24,1.00


## 4. 배치 간 입력 범위
B1 학습셀 범위 밖 B2 셀 수 — 외삽 위험 (Day 1 3-5)

In [5]:
from src.train import split_b1, in_range

# 정책 단위 분할 (학습 28 / 홀드아웃 8)
train, holdout = split_b1(feat)
b2 = feat[feat.batch == "B2"]

# 입력 구성별로, 학습 28셀의 [최솟값, 최댓값] 범위를 하나라도 벗어난 B2 셀 수
for name, cols in FEATURE_SETS.items():
    print(f"{name:20s} 범위 밖 {(~in_range(b2, train, cols)).sum():2d} / {len(b2)}")

dQ                   범위 밖 18 / 39
policy_raw           범위 밖 25 / 39
policy_derived       범위 밖  0 / 39
dQ+policy_raw        범위 밖 29 / 39
dQ+policy_derived    범위 밖 18 / 39


## 5. 정책 내부 신호와 중복

상관관계를 인과 효과로 해석하지 않고, 정책 단독·ΔQ 단독·결합 입력의 CV로 추가 정보를 확인한다.


In [6]:
centered = b1[["log_var_dQ", "cycle_life"]] - b1.groupby("policy_key")[["log_var_dQ", "cycle_life"]].transform("mean")
print("B1 종료 근접 셀 rho:", round(spearmanr(b1.log_var_dQ, b1.cycle_life)[0], 3))
print("정책 평균 제거 후 rho:", round(spearmanr(centered.log_var_dQ, centered.cycle_life)[0], 3))
b1[["log_var_dQ", "mean_dQ", "min_dQ"]].corr(method="spearman").round(3)


B1 종료 근접 셀 rho: -0.826
정책 평균 제거 후 rho: -0.116


,log_var_dQ,mean_dQ,min_dQ
log_var_dQ,1.000,-0.966,-0.988
mean_dQ,-0.966,1.000,0.980
min_dQ,-0.988,0.980,1.000
